In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import warnings

warnings.filterwarnings('ignore')

print("=" * 60)
print("🏭 Analisis Clustering Cacat Produk Industri Manufaktur")
print("=" * 60)
print("Aplikasi interaktif untuk segmentasi cacat produksi berbasis Machine Learning (K-Means Clustering).\n")

In [ ]:
# 2. Fungsi Memuat Data
def load_data():
    """Memuat data dari file CSV dengan error handling"""
    csv_path = "defects_data.csv"
    
    if not os.path.exists(csv_path):
        print(f"❌ File '{csv_path}' tidak ditemukan!")
        return None
    
    try:
        df = pd.read_csv(csv_path)
        
        # Validasi kolom yang diperlukan
        required_columns = ['defect_id', 'product_id', 'defect_type', 'severity', 'repair_cost']
        missing_cols = [col for col in required_columns if col not in df.columns]
        
        if missing_cols:
            print(f"❌ Kolom tidak lengkap: {missing_cols}")
            return None
        
        # Mapping severity ke score numerik
        severity_mapping = {'Minor': 1, 'Moderate': 2, 'Critical': 3}
        df['severity_score'] = df['severity'].map(severity_mapping)
        
        # Cek apakah ada nilai NaN setelah mapping
        if df['severity_score'].isnull().any():
            print("❌ Ada nilai severity yang tidak valid dalam data")
            return None
        
        print(f"✅ Data berhasil dimuat: {len(df)} baris")
        return df
        
    except Exception as e:
        print(f"❌ Error saat memuat data: {str(e)}")
        return None

In [ ]:
# 3. Fungsi K-Means Clustering
def perform_clustering(df, n_clusters=3):
    """Melakukan K-Means clustering dengan error handling"""
    try:
        X = df[['repair_cost', 'severity_score']]
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        kmeans_model = KMeans(
            n_clusters=n_clusters, 
            init='k-means++', 
            random_state=42, 
            n_init=10
        )
        df['cluster_kmeans'] = kmeans_model.fit_predict(X_scaled)
        
        # Hitung profil klaster
        profil_kmeans = df.groupby('cluster_kmeans')[['repair_cost', 'severity_score']].mean()
        profil_kmeans['jumlah_sampel'] = df['cluster_kmeans'].value_counts()
        
        return df, kmeans_model, scaler, profil_kmeans
        
    except Exception as e:
        print(f"❌ Error saat clustering: {str(e)}")
        return None, None, None, None

In [ ]:
# 4. Fungsi Visualisasi
def plot_clusters(df, optimal_k):
    """Membuat plot scatter untuk visualisasi klaster"""
    fig, ax = plt.subplots(figsize=(10, 6.5))
    sns.scatterplot(
        x=df['repair_cost'],
        y=df['severity_score'],
        hue=df['cluster_kmeans'],
        palette='Set1',
        s=100,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.5,
        ax=ax
    )
    ax.set_title(f'Hasil Segmentasi Kasus Cacat Menggunakan K-Means (K={optimal_k})', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Biaya Perbaikan Produk ($ / Repair Cost)', fontsize=12)
    ax.set_ylabel('Tingkat Keparahan Cacat (Severity Score)', fontsize=12)
    ax.set_yticks([1, 2, 3])
    ax.set_yticklabels(['1 (Minor)', '2 (Moderate)', '3 (Critical)'])
    ax.legend(title='Klaster K-Means', loc='upper left')
    return fig

In [ ]:
# 5. Fungsi Interpretasi Klaster
def interpret_cluster(cluster_id, row):
    """Menginterpretasikan karakteristik klaster"""
    cost = row['repair_cost']
    sev = row['severity_score']
    count = row['jumlah_sampel']
    
    if sev >= 2.3 and cost >= 200:
        kategori = "🔴 KATEGORI KRITIS & BIAYA TINGGI (High-Risk Financial Damage)"
        insight = "Kelompok ini adalah ancaman terbesar bagi profitabilitas dan reputasi perusahaan. Cacat bersifat fatal dan perbaikannya sangat mahal."
        rekomendasi = "Hentikan jalur perakitan terkait sementara untuk kalibrasi ulang komponen presisi, evaluasi vendor pemasok bahan baku utama, dan perketat lolos QC awal."
    elif sev >= 2.1:
        kategori = "🟠 KATEGORI KEPARAHAN TINGGI - BIAYA MENENGAH (Operational Issues)"
        insight = "Cacat pada kelompok ini berdampak buruk bagi fungsi produk, namun biaya penanganannya masih masuk dalam batas toleransi standar."
        rekomendasi = "Tingkatkan intensitas perawatan mesin (preventive maintenance) dan lakukan audit berkala terhadap kepatuhan SOP operator di lapangan."
    elif cost >= 150:
        kategori = "🟡 KATEGORI BIAYA TINGGI - KEPARAHAN RINGAN (Inefficient Repair Cost)"
        insight = "Tingkat keparahan produk sebenarnya tidak fatal (kosmetik/minor), namun biaya penanganan atau penggantian partnya tidak efisien."
        rekomendasi = "Tinjau ulang kontrak dengan penyedia jasa perbaikan luar atau cari komponen substitusi lokal yang lebih murah tanpa menurunkan kualitas standar."
    else:
        kategori = "🟢 KATEGORI AMAN (Low-Cost & Minor Defects)"
        insight = "Kelompok cacat ringan dengan biaya perbaikan yang sangat murah. Karakteristik ini wajar terjadi dalam batas toleransi produksi massal."
        rekomendasi = "Cukup lakukan pemantauan berkala menggunakan grafik kendali kualitas statistik (SPC) dan lakukan inspeksi visual reguler di akhir lini."
    
    return kategori, insight, rekomendasi, cost, sev, count

In [ ]:
# 6. Fungsi Prediksi Data Baru
def predict_new_cluster(kmeans_model, scaler, input_cost, input_severity):
    """Memprediksi klaster untuk data baru"""
    try:
        severity_mapping = {'Minor': 1, 'Moderate': 2, 'Critical': 3}
        severity_mapped = severity_mapping.get(input_severity, 1)
        
        new_data = np.array([[input_cost, severity_mapped]])
        new_data_scaled = scaler.transform(new_data)
        pred_cluster = kmeans_model.predict(new_data_scaled)[0]
        return pred_cluster
    except Exception as e:
        print(f"❌ Error saat prediksi: {str(e)}")
        return None

In [ ]:
# ============================================
# EKSEKUSI UTAMA
# ============================================

# Memuat data
df = load_data()

In [ ]:
if df is not None:
    # Default K untuk Jupyter
    optimal_k = 3
    
    print(f"\n📊 Melakukan clustering dengan K={optimal_k}...")
    df, kmeans_model, scaler, profil_kmeans = perform_clustering(df, optimal_k)

In [ ]:
if kmeans_model is not None:
    # Visualisasi
    print("\n📈 Menampilkan visualisasi klaster...")
    fig = plot_clusters(df, optimal_k)
    plt.show()

In [ ]:
if kmeans_model is not None:
    # Statistik
    print("\n📊 Statistik Rata-rata Tiap Klaster:")
    display(profil_kmeans[['repair_cost', 'severity_score']].style.format({
        "repair_cost": "${:.2f}", 
        "severity_score": "{:.2f}"
    }))

In [ ]:
if kmeans_model is not None:
    print(f"\n📊 Jumlah Sampel per Klaster:")
    print(df['cluster_kmeans'].value_counts())

In [ ]:
if kmeans_model is not None:
    # Interpretasi
    print("\n💡 Interpretasi & Insight Bisnis Mendalam")
    print("=" * 60)
    
    for cluster_id, row in profil_kmeans.iterrows():
        kategori, insight, rekomendasi, cost, sev, count = interpret_cluster(cluster_id, row)
        
        print(f"\nKlaster {cluster_id}: {kategori}")
        print(f"  - Rata-rata Biaya Perbaikan: ${cost:.2f}")
        print(f"  - Skor Keparahan (1-3): {sev:.2f}")
        print(f"  - Total Temuan Kasus: {int(count)} Produk")
        print(f"  - Analisis Masalah: {insight}")
        print(f"  - Rekomendasi Strategis: {rekomendasi}")
        print("-" * 60)

In [ ]:
if kmeans_model is not None:
    # Contoh Prediksi
    print("\n🛠️ Contoh Prediksi Data Baru:")
    print("-" * 60)
    
    test_cases = [
        (50.0, 'Minor'),
        (100.0, 'Moderate'),
        (300.0, 'Critical')
    ]
    
    for cost, severity in test_cases:
        pred_cluster = predict_new_cluster(kmeans_model, scaler, cost, severity)
        if pred_cluster is not None:
            print(f"  Biaya: ${cost:.2f}, Severity: {severity} -> Klaster {pred_cluster}")
    
    print("\n✅ Analisis selesai!")
else:
    print("\n⚠️ Tidak dapat melanjutkan karena data gagal dimuat.")